In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


from surprise import accuracy

# class is used to parse a file containing ratings, data should be in structure - user ; item ; rating
from surprise.reader import Reader

# class for loading datasets
from surprise.dataset import Dataset

# for model tuning model hyper-parameters
from surprise.model_selection import GridSearchCV

# for splitting the rating data in train and test dataset
from surprise.model_selection import train_test_split

# for implementing similarity based recommendation system
from surprise.prediction_algorithms.knns import KNNBasic

# for implementing matrix factorization based recommendation system
from surprise.prediction_algorithms.matrix_factorization import SVD

from collections import defaultdict

# for implementing cross validation
from surprise.model_selection import KFold


In [2]:
ratings=pd.read_csv('rate.csv')

In [3]:
ratings.info

<bound method DataFrame.info of        userId  movieId  rating   timestamp
0           2       17     5.0   835355681
1           2      153     4.0   835355441
2           2      186     3.0   835355664
3           2      272     3.0   835355767
4           2      292     3.0   835355492
...       ...      ...     ...         ...
12931     671     2959     4.0  1064890427
12932     671     4880     4.0  1065111973
12933     671     4896     5.0  1065111996
12934     671     5816     4.0  1065111963
12935     671     5995     4.0  1066793014

[12936 rows x 4 columns]>

In [4]:
ratings.shape

(12936, 4)

In [5]:
ratings.head(5)

,userId,movieId,rating,timestamp
0,2,17,5.0,835355681
1,2,153,4.0,835355441
2,2,186,3.0,835355664
3,2,272,3.0,835355767
4,2,292,3.0,835355492


In [6]:
ratings['userId'].nunique() 

665

In [7]:
ratings['movieId'].nunique()

870

In [8]:
ratings.groupby(['userId', 'movieId']).count()

rating  timestamp
userId movieId                   
2      17            1          1
       153           1          1
       186           1          1
       272           1          1
       292           1          1
...                ...        ...
671    2959          1          1
       4880          1          1
       4896          1          1
       5816          1          1
       5995          1          1

[12936 rows x 2 columns]

In [9]:
ratings.groupby(['userId', 'movieId']).count()['rating'].sum()

12936

In [10]:
ratings= ratings.drop(['timestamp'], axis=1)

In [11]:
# instantiating Reader scale with expected rating scale
reader = Reader(rating_scale=(0, 5))

# loading the rating dataset
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# splitting the data into train and test dataset
trainset, testset = train_test_split(data, test_size=0.2, random_state=40)

In [12]:
# using SVD matrix factorization
algo_svd = SVD()

# training the algorithm on the trainset
algo_svd.fit(trainset)

# predicting ratings for the testset
predictions = algo_svd.test(testset)

# computing RMSE on the testset
print(accuracy.rmse(predictions))
print(accuracy.mae(predictions))

RMSE: 0.9579
0.9578805316410248
MAE:  0.7378
0.7378494162446304


In [13]:
 #predict rating for an user with userId=4 and for movieId=10
algo_svd.predict(4, 10, r_ui=4, verbose=True)    

user: 4          item: 10         r_ui = 4.00   est = 4.05   {'was_impossible': False}


Prediction(uid=4, iid=10, r_ui=4, est=4.050761011246434, details={'was_impossible': False})

In [14]:
#Let's predict the rating for the same userId=4 but for a movie which this user has not interacted before i.e. movieId=3

algo_svd.predict(4, 3, verbose=True)

user: 4          item: 3          r_ui = None   est = 4.05   {'was_impossible': False}


Prediction(uid=4, iid=3, r_ui=None, est=4.050761011246434, details={'was_impossible': False})

In [15]:
# set the parameter space to tune
param_grid = {'n_epochs': [10,20,30], 'lr_all': [0.001, 0.005, 0.01],
              'reg_all': [0.06, 0.08, 0.006]}

# performing 3-fold gridsearch cross validation
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)

# fitting data
gs.fit(data)
para=[]
# best RMSE score
print(gs.best_score['rmse'])
print(gs.best_score['mae'])
# combination of parameters that gave the best RMSE score
para=gs.best_params['rmse']

0.9150724615372857
0.7074083814456352


In [16]:
para

{'n_epochs': 20, 'lr_all': 0.01, 'reg_all': 0.08}

In [17]:
results_df = pd.DataFrame.from_dict(gs.cv_results)
results_df.head()

,split0_test_rmse,split1_test_rmse,split2_test_rmse,mean_test_rmse,std_test_rmse,rank_test_rmse,split0_test_mae,split1_test_mae,split2_test_mae,mean_test_mae,std_test_mae,rank_test_mae,mean_fit_time,std_fit_time,mean_test_time,std_test_time,params,param_n_epochs,param_lr_all,param_reg_all
0,0.993074,0.979637,0.976274,0.982995,0.007258,26,0.787305,0.774990,0.771353,0.777882,0.006826,26,0.092766,1.406581e-03,0.052088,0.007368,"{'n_epochs': 10, 'lr_all': 0.001, 'reg_all': 0...",10,0.001,0.060
1,0.991437,0.982811,0.977387,0.983878,0.005785,27,0.786302,0.777101,0.772168,0.778524,0.005857,27,0.093756,5.150430e-07,0.046881,0.000002,"{'n_epochs': 10, 'lr_all': 0.001, 'reg_all': 0...",10,0.001,0.080
2,0.993250,0.979370,0.972816,0.981812,0.008519,25,0.785994,0.773400,0.768816,0.776070,0.007263,25,0.109699,2.213526e-04,0.047469,0.000507,"{'n_epochs': 10, 'lr_all': 0.001, 'reg_all': 0...",10,0.001,0.006
3,0.944844,0.928861,0.921813,0.931839,0.009635,16,0.732743,0.721847,0.712824,0.722471,0.008144,17,0.105119,7.711840e-03,0.036520,0.008229,"{'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0...",10,0.005,0.060
4,0.941808,0.929687,0.921925,0.931140,0.008182,15,0.729028,0.722010,0.713284,0.721441,0.006440,15,0.104855,7.778749e-03,0.036139,0.006814,"{'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0...",10,0.005,0.080


In [18]:
# building the optimized SVD model using optimal hyperparameter search
svd_algo_optimized = SVD(n_epochs=para['n_epochs'], lr_all=para['lr_all'], reg_all=para['reg_all'])

# training the algorithm on the trainset
svd_algo_optimized.fit(trainset)

# predicting ratings for the testset
predictions = svd_algo_optimized.test(testset)

# computing RMSE
accuracy.rmse(predictions)

RMSE: 0.9548


0.9548396306443687

In [19]:
algo_svd.predict(4,10, r_ui=4, verbose=True)

user: 4          item: 10         r_ui = 4.00   est = 4.05   {'was_impossible': False}


Prediction(uid=4, iid=10, r_ui=4, est=4.050761011246434, details={'was_impossible': False})

In [20]:
algo_svd.predict(4, 7, verbose=True)

user: 4          item: 7          r_ui = None   est = 4.05   {'was_impossible': False}


Prediction(uid=4, iid=7, r_ui=None, est=4.050761011246434, details={'was_impossible': False})

In [21]:
def get_recommendations(data, user_id, top_n, algo):
    
    # creating an empty list to store the recommended movie ids
    recommendations = []
    
    # creating an user item interactions matrix 
    user_item_interactions_matrix = data.pivot(index='userId', columns='movieId', values='rating')
    
    # extracting those movie ids which the user_id has not interacted yet
    non_interacted_movies = user_item_interactions_matrix.loc[user_id][user_item_interactions_matrix.loc[user_id].isnull()].index.tolist()
    
    # looping through each of the movie id which user_id has not interacted yet
    for item_id in non_interacted_movies:
        
        # predicting the ratings for those non interacted movie ids by this user
        est = algo.predict(user_id, item_id).est
        
        # appending the predicted ratings
        recommendations.append((item_id, est))

    # sorting the predicted ratings in descending order
    recommendations.sort(key=lambda x: x[1], reverse=True)

    return recommendations[:top_n] # returing top n highest predicted rating movies for this user

In [22]:
get_recommendations(ratings, 564, 10, svd_algo_optimized)

[(1944, 4.17507689569045),
 (6016, 4.172088935367593),
 (52767, 4.168012907041002),
 (1254, 4.133255829250543),
 (89492, 4.129359524984758),
 (26587, 4.113388546641218),
 (308, 4.108571733095303),
 (3037, 4.093728512660846),
 (8827, 4.087185755076613),
 (48780, 4.083193152385718)]

In [ ]:
def train_collaborative(ratings):
    reader = Reader(rating_scale=(0, 5))

    # loading the rating dataset
    data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

    # splitting the data into train and test dataset
    trainset, testset = train_test_split(data, test_size=0.2, random_state=40)
    # using SVD matrix factorization
    algo_svd = SVD()

    # training the algorithm on the trainset
    algo_svd.fit(trainset)

    # predicting ratings for the testset
    predictions = algo_svd.test(testset)

    # computing RMSE on the testset
    print(accuracy.rmse(predictions))
    print(accuracy.mae(predictions))

    # set the parameter space to tune
    param_grid = {'n_epochs': [10,20,30], 'lr_all': [0.001, 0.005, 0.01],
              'reg_all': [0.06, 0.08, 0.006]}

    # performing 3-fold gridsearch cross validation
    gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)

    # fitting data
    gs.fit(data)
    para=[]
    para=gs.best_params['rmse']
    # building the optimized SVD model using optimal hyperparameter search
    svd_algo_optimized = SVD(n_epochs=para['n_epochs'], lr_all=para['lr_all'], reg_all=para['reg_all'])

    # training the algorithm on the trainset
    svd_algo_optimized.fit(trainset)

    # predicting ratings for the testset
    predictions = svd_algo_optimized.test(testset)

    # computing RMSE
    accuracy.rmse(predictions)